In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from torch.optim import Adam

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- THAY ĐỔI SIÊU THAM SỐ: LEARNING RATE = 0.00001 (Mức quá thấp) ---
def get_model_low_lr():
    model = nn.Sequential(
        nn.Linear(28 * 28, 1000),
        nn.ReLU(),
        nn.Linear(1000, 10)
    ).to(device)
    
    loss_fn = nn.CrossEntropyLoss()
    # Thiết lập lr=1e-5 (tức là 0.00001) 
    optimizer = Adam(model.parameters(), lr=1e-5)
    return model, loss_fn, optimizer

In [ ]:
# Đảm bảo dữ liệu trn_dl và val_dl từ các bài trước đã sẵn sàng
trn_dl, val_dl = get_data()

model, loss_fn, optimizer = get_model_low_lr()
train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

for epoch in range(5):
    print(f"Epoch đang chạy (Adam với LR = 0.00001): {epoch}")
    train_epoch_losses, train_epoch_accuracies = [], []
    
    # 1. Huấn luyện cập nhật trọng số trên tập Train
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        batch_loss = train_batch(x, y, model, optimizer, loss_fn)
        train_epoch_losses.append(batch_loss)
    train_epoch_loss = np.array(train_epoch_losses).mean()
    
    # 2. Tính độ chính xác tập Train
    for ix, batch in enumerate(iter(trn_dl)):
        x, y = batch
        is_correct = accuracy(x, y, model)
        train_epoch_accuracies.extend(is_correct)
    train_epoch_accuracy = np.mean(train_epoch_accuracies)
    
    # 3. Đánh giá trên tập Validation
    for ix, batch in enumerate(iter(val_dl)):
        x, y = batch
        val_loss_value = val_loss(x, y, model, loss_fn)
        val_is_correct = accuracy(x, y, model)
    val_epoch_accuracy = np.mean(val_is_correct)
    
    # Lưu trữ kết quả lịch sử
    train_losses.append(train_epoch_loss)
    train_accuracies.append(train_epoch_accuracy)
    val_losses.append(val_loss_value)
    val_accuracies.append(val_epoch_accuracy)

In [ ]:
epochs = np.arange(5) + 1
plt.figure(figsize=(10, 10))

# Biểu đồ Loss (Training vs Validation với LR = 0.00001)
plt.subplot(211)
plt.plot(epochs, train_losses, 'bo', label='Training loss')
plt.plot(epochs, val_losses, 'r', label='Validation loss')
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.title('Training and validation loss with 0.00001 learning rate')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(False)

# Biểu đồ Accuracy (Training vs Validation với LR = 0.00001)
plt.subplot(212)
plt.plot(epochs, train_accuracies, 'bo', label='Training accuracy')
plt.plot(epochs, val_accuracies, 'r', label='Validation accuracy')
plt.gca().xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.title('Training and validation accuracy with 0.00001 learning rate')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.gca().set_yticklabels(['{:.0f}%'.format(x * 100) for x in plt.gca().get_yticks()])
plt.legend()
plt.grid(False)

plt.tight_layout()
plt.show()